# 02 — Track A: fine-tune BiomedCLIP on image–text pairs (Phase 6)

GPU steps only: fine-tune → re-encode images with the adapted model → zero-shot.
Download the new embeddings folder afterwards; the linear heads are trained locally.

Same one-time setup as notebook 01 (T4 GPU, `GITHUB_TOKEN` secret, data on Drive).

**Experiment:** set `RUN` below. Step 1 is `a1` — adapt on the 119 real expert captions only
(other clinic, no Kromp labels), then check zero-shot grading and (locally) the linear heads.

In [ ]:
# Experiment settings
RUN = 'a1'        # a1 = real captions only | a2 = captions + pseudo-captions of N labelled images
SEED = 0
N_LABELS = None   # a2 only: number of labelled Kromp training images (e.g. 50, 100, 200, 500)
EPOCHS = 20

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/blastocyst-clip'
import os

In [ ]:
# 3. Get the latest code
from google.colab import userdata
REPO = '/content/blastocyst-clip'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/Daniel0813/blastocyst-clip.git'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q {url} {REPO}
del token, url
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Install
!pip install -q open_clip_torch
import torch, open_clip, timm, sys
print('torch', torch.__version__, '| open_clip', open_clip.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 5. Rebuild the caption subset on Drive (adds the fixed held-out split used for cross-clinic testing)
!python src/select_blastocyst_captions.py --captions-dir {DRIVE}/data/captions

In [ ]:
# 6. Fine-tune (only the last 2 blocks of each encoder train; ~1-3 min on a T4)
NAME = f'biomedclip_{RUN}' + (f'_n{N_LABELS}' if N_LABELS else '') + f'_seed{SEED}'
FT_DIR = f'{DRIVE}/results/finetune/{NAME}'
mode = {'a1': 'captions', 'a2': 'both'}[RUN]
extra = f'--n-labels {N_LABELS}' if N_LABELS else ''
!python -W ignore src/finetune.py --data-dir {DRIVE}/data --mode {mode} {extra} --seed {SEED} --epochs {EPOCHS} --out-dir {FT_DIR}

In [ ]:
# 7. Re-encode all images with the adapted model (all three feature types)
EMB_DIR = f'{DRIVE}/results/embeddings/{NAME}'
!python -W ignore src/embed.py --data-dir {DRIVE}/data --weights {FT_DIR}/adapter.pt --out-dir {EMB_DIR}

In [ ]:
# 8. Zero-shot grading with the adapted model (frozen BiomedCLIP scored 0.134 / 0.168 / 0.184)
!python -W ignore src/zeroshot.py --emb-dir {EMB_DIR} --labels {DRIVE}/data/blastocyst/labels.csv

In [ ]:
# 9. Record provenance
import json, subprocess, datetime
info = {'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip(),
        'date': datetime.datetime.now().isoformat(timespec='seconds'), 'run': NAME,
        'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
json.dump(info, open(f'{EMB_DIR}/run_info.json', 'w'), indent=2)
print(info)
print('Download this folder for the local linear-head step:', EMB_DIR)